In [13]:
import sqlite3
import pandas as pd
from datetime import datetime

# connecties met sdm en dwh databases (brondatabase + dwh database afgeleid van ETL-schema's)

# bron databases
conn_acc_verkoop = sqlite3.connect("BikeToDrive_1_Accessoireverkoop.db")
conn_fiets_verkoop = sqlite3.connect("BikeToDrive_2_Fietsverkoop.db")
conn_onderhoud = sqlite3.connect("BikeToDrive_3_Onderhoud.db")
conn_acc_inkoop = sqlite3.connect("BikeToDrive_4_Accessoire_Inkoop.db")
conn_fiets_inkoop = sqlite3.connect("BikeToDrive_5_Fiets_Inkoop.db")

# data warehouse
dwh_conn = sqlite3.connect("DWH_DB.db")

In [12]:
# als er iets mis is met de connection of foutjes in database run dit zodat de connection wordt gestopt en je opnieuw kan proberen !

sdm_conn.close()

In [ ]:
# etl: extract (pak de data die je nodig hebt basically)
verkoop_acc = pd.read_sql("SELECT * FROM Accessoire_Verkoop", conn_acc_verkoop)
verkoop_fiets = pd.read_sql("SELECT * FROM Verkoop", conn_fiets_verkoop)

# etl: transform (gepakte data combineren)
verkoop_all = pd.concat([verkoop_acc, verkoop_fiets], ignore_index=True)

# etl: load (laadt alleen nieuwe records)
dwh_verkoop = pd.read_sql("SELECT * FROM Verkoop", dwh_conn)

nieuwe_verkoop = verkoop_all[
    ~verkoop_all['verkoopnr'].isin(dwh_verkoop['verkoopnr'])
]

nieuwe_verkoop.to_sql("Verkoop", dwh_conn, if_exists='append', index=False)

DatabaseError: Execution failed

In [ ]:
def scd_type1(table, key, source_conn):
    # extract uit source
    sdm = pd.read_sql(f"SELECT * FROM {table}", source_conn)
    
    # bestaande data in DWH
    dwh = pd.read_sql(f"SELECT * FROM {table}", dwh_conn)

    for _, row in sdm.iterrows():
        exists = dwh[dwh[key] == row[key]]

        if exists.empty:
            # insert in nieuwe rij
            row.to_frame().T.to_sql(table, dwh_conn, if_exists='append', index=False)
        else:
            # bestaande rij wordt geupdate (overgeschreven)
            set_clause = ", ".join([f"{col}=?" for col in row.index])
            values = list(row.values)

            dwh_conn.execute(
                f"UPDATE {table} SET {set_clause} WHERE {key}=?",
                values + [row[key]]
            )

    dwh_conn.commit()

In [ ]:
scd_type1("Leverancier", "leveranciernr", conn_acc_inkoop)
scd_type1("Fabrikant", "fabrikantnr", conn_acc_inkoop)
scd_type1("Monteur", "monteurnr", conn_onderhoud)
scd_type1("Filiaal", "filiaalnr", conn_onderhoud)

In [ ]:
def scd_type2(table, key, source_conn):
    # extract
    sdm = pd.read_sql(f"SELECT * FROM {table}", source_conn)
    dwh = pd.read_sql(f"SELECT * FROM {table}", dwh_conn)

    for _, row in sdm.iterrows():
        # huidige actieve rij in DWH
        current = dwh[(dwh[key] == row[key]) & (dwh['is_current'] == 1)]

        if current.empty:
            # nieuwe record
            row_dict = row.to_dict()
            row_dict['valid_from'] = datetime.now()
            row_dict['valid_to'] = None
            row_dict['is_current'] = 1

            pd.DataFrame([row_dict]).to_sql(table, dwh_conn, if_exists='append', index=False)

        else:
            current_row = current.iloc[0]

            # check of er iets veranderd is
            changed = any(row[col] != current_row[col] for col in row.index)

            if changed:
                # oude record afsluiten
                dwh_conn.execute(
                    f"""
                    UPDATE {table}
                    SET valid_to = ?, is_current = 0
                    WHERE {key} = ? AND is_current = 1
                    """,
                    (datetime.now(), row[key])
                )

                # nieuwe versie toevoegen
                row_dict = row.to_dict()
                row_dict['valid_from'] = datetime.now()
                row_dict['valid_to'] = None
                row_dict['is_current'] = 1

                pd.DataFrame([row_dict]).to_sql(table, dwh_conn, if_exists='append', index=False)

    dwh_conn.commit()

In [ ]:
scd_type2("Klant", "klantnr", conn_fiets_verkoop)
scd_type2("Product", "productnr", conn_fiets_verkoop)

In [ ]:
def load_fact(table, source_conn):
    sdm = pd.read_sql(f"SELECT * FROM {table}", source_conn)
    dwh = pd.read_sql(f"SELECT * FROM {table}", dwh_conn)

    nieuwe = sdm[~sdm.iloc[:,0].isin(dwh.iloc[:,0])]

    nieuwe.to_sql(table, dwh_conn, if_exists='append', index=False)

# uitvoeren
load_fact("Inkoop", conn_acc_inkoop)
load_fact("Onderhoud", conn_onderhoud)